# Manufacturing Multi-Agent — Evaluation & Cost Tracking

End-to-end evaluation of the multi-agent system:
1. **Classifier eval** — Accuracy scorecard + confusion matrix
2. **End-to-end eval** — Supervisor routing + answer quality
3. **Cost tracking** — System tables billing analysis (optional; needs access to `system.billing.usage`)

**Databricks AI Features:** MLflow, Foundation Models, system.billing.usage

Default sample size is 50 inspections. Raise `eval_sample_size` toward 500 only after you accept the extra model cost.


Install LangChain packages first. `%pip` restarts Python, so it must run before `_config` sets up widgets and `sys.path`.


In [ ]:
%pip install databricks-langchain langchain-core -q


## Setup


In [ ]:
%run ./_config


Re-read job widgets in this notebook so catalog and schema are defined after `%run`.


In [ ]:
# Job parameters / widgets must be read in this notebook. %run does not always export names from .ipynb.
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
try:
    llm_endpoint = dbutils.widgets.get("llm_endpoint").strip()
except Exception:
    llm_endpoint = "databricks-claude-sonnet-4-5"
try:
    embedding_endpoint = dbutils.widgets.get("embedding_endpoint").strip()
except Exception:
    embedding_endpoint = "databricks-gte-large-en"
try:
    vs_endpoint = dbutils.widgets.get("vs_endpoint").strip()
except Exception:
    vs_endpoint = "mfg_quality_vs_endpoint"
try:
    eval_sample_size = dbutils.widgets.get("eval_sample_size").strip() or "50"
except Exception:
    eval_sample_size = "50"
try:
    cost_owner = dbutils.widgets.get("cost_owner").strip()
except Exception:
    cost_owner = ""
VS_INDEX_NAME = f"{catalog}.{schema}.service_bulletins_index"
print(f"Bound widgets: {catalog}.{schema}")


Load SQL guardrails and supervisor `run_agent`.


Build the shared supervisor (`run_agent`) from `src/agent_runtime.py`.


In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks
from agent_runtime import build_supervisor

spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")
sample_n = max(1, min(500, int(eval_sample_size)))
print(f"Classifier sample size: {sample_n}")

def execute_sql(sql_text):
    return spark.sql(sql_text).limit(20).toPandas().to_dict(orient="records")

llm = ChatDatabricks(endpoint=llm_endpoint, temperature=0.1)
supervisor = build_supervisor(
    llm=llm,
    catalog=catalog,
    schema=schema,
    vs_index=VS_INDEX_NAME,
    execute_sql=execute_sql,
    workspace_client=WorkspaceClient(),
)
run_agent = supervisor.run


## 1. Classifier Eval — Scorecard
Runs `ai_query` on a sample of inspections, then scores severity and root cause.


In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE classifier_results AS
SELECT
  inspection_id, part_type, plant_name, line,
  defect_notes, expected_severity, expected_root_cause,
  ai_query(
    '{llm_endpoint}',
    CONCAT(
      'Classify this manufacturing inspection. Return ONLY JSON:
{{"severity": "Critical|Major|Minor|None", "root_cause": "Material|Process|Equipment|Human|None", "corrective_action": "one sentence"}}

INSPECTION: ', defect_notes)
  ) AS ai_output
FROM {catalog}.{schema}.quality_inspections
LIMIT {sample_n}
""")


Score parsed labels. Empty regexp matches are treated as missing so keyword fallbacks can run.


In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE classifier_scored AS
WITH parsed AS (
  SELECT *,
    CASE
      WHEN ai_output RLIKE '"severity"\\\\s*:\\\\s*"(Critical|Major|Minor|None)"'
        THEN regexp_extract(ai_output, '"severity"\\\\s*:\\\\s*"(Critical|Major|Minor|None)"', 1)
      WHEN lower(ai_output) RLIKE '(critical|safety|quarantine)' THEN 'Critical'
      WHEN lower(ai_output) RLIKE '(major|reject|rework)' THEN 'Major'
      WHEN lower(ai_output) RLIKE '(minor|borderline|monitor)' THEN 'Minor'
      WHEN lower(ai_output) RLIKE '(none|pass|approved|within)' THEN 'None'
      ELSE 'UNPARSEABLE'
    END AS parsed_severity,
    CASE
      WHEN ai_output RLIKE '"root_cause"\\\\s*:\\\\s*"(Material|Process|Equipment|Human|None)"'
        THEN regexp_extract(ai_output, '"root_cause"\\\\s*:\\\\s*"(Material|Process|Equipment|Human|None)"', 1)
      WHEN lower(ai_output) RLIKE '(material|supplier)' THEN 'Material'
      WHEN lower(ai_output) RLIKE '(process|temperature|pressure)' THEN 'Process'
      WHEN lower(ai_output) RLIKE '(equipment|tool|calibrat)' THEN 'Equipment'
      WHEN lower(ai_output) RLIKE '(human|operator|wrong)' THEN 'Human'
      ELSE 'None'
    END AS parsed_root_cause
  FROM classifier_results
)
SELECT *,
  (parsed_severity = expected_severity) AS severity_correct,
  (parsed_root_cause = expected_root_cause) AS root_cause_correct,
  (parsed_severity = expected_severity AND parsed_root_cause = expected_root_cause) AS both_correct
FROM parsed
""")


Overall scorecard.


In [ ]:
display(spark.sql("""
SELECT
  COUNT(*) AS total_inspections,
  ROUND(AVG(CAST(severity_correct AS INT)) * 100, 1) AS severity_accuracy_pct,
  ROUND(AVG(CAST(root_cause_correct AS INT)) * 100, 1) AS root_cause_accuracy_pct,
  ROUND(AVG(CAST(both_correct AS INT)) * 100, 1) AS both_correct_pct,
  SUM(CASE WHEN parsed_severity = 'UNPARSEABLE' THEN 1 ELSE 0 END) AS unparseable
FROM classifier_scored
"""))


Severity confusion matrix.


In [ ]:
display(spark.sql("""
SELECT expected_severity, parsed_severity, COUNT(*) AS cnt
FROM classifier_scored
GROUP BY expected_severity, parsed_severity
ORDER BY expected_severity, parsed_severity
"""))


Accuracy by part type.


In [ ]:
display(spark.sql("""
SELECT
  part_type,
  COUNT(*) AS n,
  ROUND(AVG(CAST(severity_correct AS INT)) * 100, 1) AS severity_pct,
  ROUND(AVG(CAST(both_correct AS INT)) * 100, 1) AS both_pct
FROM classifier_scored
GROUP BY part_type
ORDER BY both_pct ASC
"""))


## 2. End-to-End Agent Eval with MLflow


In [ ]:
import mlflow
import pandas as pd

eval_data = pd.DataFrame([
    {"question": "Brake rotor measured 22.5mm on Line 2, spec is 28-32mm. What severity?",
     "expected_answer_contains": "Critical", "query_type": "classification"},
    {"question": "Door panel gap at 1.3mm, spec 1.0-1.5mm. Any concern?",
     "expected_answer_contains": "None", "query_type": "classification"},
    {"question": "Paint surface at 65 microns, spec 80-120. What should I do?",
     "expected_answer_contains": "Major", "query_type": "classification"},
    {"question": "What service bulletins exist for brake rotor thickness issues?",
     "expected_answer_contains": "SB-", "query_type": "retrieval"},
    {"question": "Any known fixes for wiring harness intermittent failures?",
     "expected_answer_contains": "SB-", "query_type": "retrieval"},
    {"question": "What's the procedure for exhaust manifold thermal cracking?",
     "expected_answer_contains": "SB-", "query_type": "retrieval"},
    {"question": "What's the average FTQ rate across all plants this month?",
     "expected_answer_contains": "ftq", "query_type": "sql"},
    {"question": "Top 3 root causes for critical defects at Arlington Assembly?",
     "expected_answer_contains": "Equipment", "query_type": "sql"},
    {"question": "Total rework cost by plant for the last quarter?",
     "expected_answer_contains": "$", "query_type": "sql"},
    {"question": "How many open corrective actions are there?",
     "expected_answer_contains": "action", "query_type": "sql"},
    {"question": "We found a battery module at 2.8V on Line 4 at Spring Hill. Is this a known issue and what's the trend?",
     "expected_answer_contains": "Critical", "query_type": "hybrid"},
    {"question": "Transmission gear tooth profile deviation at Flint. What bulletins apply and how many similar defects this quarter?",
     "expected_answer_contains": "SB-", "query_type": "hybrid"},
])

print(f"Eval dataset: {len(eval_data)} questions")


Run the supervisor on the eval questions.


In [ ]:
results = []
for _, row in eval_data.iterrows():
    try:
        answer = run_agent(row["question"])
        contains_expected = row["expected_answer_contains"].lower() in answer.lower()
        results.append({
            "question": row["question"],
            "query_type": row["query_type"],
            "answer": answer[:500],
            "contains_expected": contains_expected,
        })
    except Exception as e:
        results.append({
            "question": row["question"],
            "query_type": row["query_type"],
            "answer": f"ERROR: {e}",
            "contains_expected": False,
        })

eval_results = pd.DataFrame(results)

print("=== Agent Eval Scorecard ===\n")
overall = eval_results["contains_expected"].mean() * 100
print(f"Overall accuracy: {overall:.1f}%\n")

for qt in ["classification", "retrieval", "sql", "hybrid"]:
    subset = eval_results[eval_results.query_type == qt]
    acc = subset["contains_expected"].mean() * 100
    print(f"  {qt:15s}: {acc:.1f}% ({subset['contains_expected'].sum()}/{len(subset)})")

with mlflow.start_run(run_name="agent_eval"):
    mlflow.log_metric("overall_accuracy", float(overall))
    for qt in ["classification", "retrieval", "sql", "hybrid"]:
        subset = eval_results[eval_results.query_type == qt]
        mlflow.log_metric(f"{qt}_accuracy", float(subset["contains_expected"].mean() * 100))
    mlflow.log_table(eval_results, "eval_results.json")
    print(f"\nEval logged to MLflow run: {mlflow.active_run().info.run_id}")


## 3. Cost Tracking via System Tables
Requires `SELECT` on `system.billing.usage`. Billing rows can lag 1-4 hours. This step is optional and is skipped if you do not have access.


In [ ]:
owner = cost_owner or spark.sql("SELECT current_user() AS u").first()["u"]
print(f"Cost filter identity: {owner}")

try:
    display(spark.sql(f"""
    SELECT
      billing_origin_product,
      sku_name,
      ROUND(SUM(usage_quantity), 4) AS total_dbus
    FROM system.billing.usage
    WHERE usage_start_time >= DATEADD(DAY, -1, CURRENT_TIMESTAMP())
      AND (
        custom_tags['Owner'] = '{owner}'
        OR identity_metadata.run_as = '{owner}'
      )
      AND billing_origin_product IN ('MODEL_SERVING', 'VECTOR_SEARCH', 'SQL')
    GROUP BY billing_origin_product, sku_name
    ORDER BY total_dbus DESC
    """))
except Exception as e:
    print(f"Billing tables not available or query failed: {e}")
    print("Grant SELECT on system.billing.usage, or skip this section.")


## Summary


In [ ]:
print("Evaluation complete:")
print("  1. Classifier scorecard")
print("  2. Agent eval (12 questions across 4 types)")
print("  3. Optional cost tracking via system.billing.usage")
print()
print("Next: open the Databricks App mfg-quality-assistant")
